In [1]:
import numpy as np
import os, json, random, zipfile
os.environ["KERAS_BACKEND"] = "tensorflow"
import keras
import tensorflow as tf

SEED    = 42
N_TEST  = 100     # first 100 MNIST test images (not hand-picked)
SHIFT   = 24      # S in the requantization formula
OUT_DIR = "mnist_cnn_data"

random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

In [2]:
#get data
(train_images, train_labels), (test_images, test_labels) = \
      keras.datasets.mnist.load_data()

#scale
train_images = train_images / 255.0
test_images  = test_images / 255.0

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [3]:
#setup model (small CNN, sized for 4x4 tiles: 8 filters = 2 tiles wide)
model = keras.Sequential(
    [
    keras.Input(shape=(28, 28)),
    keras.layers.Reshape(target_shape=(28, 28, 1)),

    keras.layers.Conv2D(kernel_size=3, filters=8, padding='valid', activation='relu'),
    keras.layers.MaxPooling2D(pool_size=2),

    keras.layers.Conv2D(kernel_size=3, filters=8, padding='valid', activation='relu'),
    keras.layers.MaxPooling2D(pool_size=2),

    keras.layers.Flatten(),
    keras.layers.Dense(10)          # logits; argmax gives the digit
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape (Reshape)               │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 26, 26, 8)      │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 8)      │           584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 8)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │         2,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,674 (10.45 KB)

 Trainable params: 2,674 (10.45 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
#compile and train
model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])
model.fit(train_images, train_labels, epochs=5, batch_size=64)

#evaluate float model
test_loss, float_acc = model.evaluate(test_images, test_labels, verbose=0)
print('Float test accuracy (10,000 images):', float_acc)

Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 20s 20ms/step - accuracy: 0.8753 - loss: 0.4317
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 22s 22ms/step - accuracy: 0.9556 - loss: 0.1491
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 19s 20ms/step - accuracy: 0.9655 - loss: 0.1150
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 20s 21ms/step - accuracy: 0.9711 - loss: 0.0966
Epoch 5/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 18s 19ms/step - accuracy: 0.9748 - loss: 0.0845
Float test accuracy (10,000 images): 0.9764999747276306


In [5]:
conv1, conv2, dense = [l for l in model.layers if l.get_weights()]
W1, b1 = conv1.get_weights()     # (3,3,1,8), (8,)
W2, b2 = conv2.get_weights()     # (3,3,8,8), (8,)
W3, b3 = dense.get_weights()     # (200,10), (10,)

def qweight(W):
    s = np.max(np.abs(W)) / 127.0                         # one scale per layer
    q = np.clip(np.round(W / s), -127, 127).astype(np.int64)
    return q, s

def quant_input(x):              # pixels 0.0-1.0 -> int8 0..127
    return np.clip(np.round(x * 127), 0, 127).astype(np.int64)

s_in = 1.0 / 127
W1_q, s_w1 = qweight(W1)
W2_q, s_w2 = qweight(W2)
W3_q, s_w3 = qweight(W3)
print("weight scales:", s_w1, s_w2, s_w3)

weight scales: 0.005322765 0.0065714438 0.0074067814


In [6]:
# How big do activations get? Measure on 2000 training images.
act_model = keras.Model(model.inputs,
                        [model.layers[2].output,    # after conv1 + pool
                         model.layers[4].output])   # after conv2 + pool
a1_f, a2_f = act_model.predict(train_images[:2000], verbose=0)
s_a1 = float(np.max(a1_f)) / 127.0
s_a2 = float(np.max(a2_f)) / 127.0

# Biases become int32, at the same scale as the accumulator
b1_q = np.round(b1 / (s_in * s_w1)).astype(np.int64)
b2_q = np.round(b2 / (s_a1 * s_w2)).astype(np.int64)
b3_q = np.round(b3 / (s_a2 * s_w3)).astype(np.int64)

# Integer multipliers for requantization: out = (acc * M) >> SHIFT
M1 = int(round((s_in * s_w1 / s_a1) * (1 << SHIFT)))
M2 = int(round((s_a1 * s_w2 / s_a2) * (1 << SHIFT)))
assert 0 < M1 < (1 << 31) and 0 < M2 < (1 << 31), "M out of range"
print("M1 =", M1, " M2 =", M2, " SHIFT =", SHIFT)

W1_mat = W1_q.reshape(-1, 8)     # (9, 8)    row k = (kh*3 + kw)*Cin + cin
W2_mat = W2_q.reshape(-1, 8)     # (72, 8)
W3_mat = W3_q                    # (200, 10)

M1 = 33088  M2 = 32130  SHIFT = 24


In [7]:
def im2col(x, k=3):
    """x: (N,H,W,C) -> patches (N, Ho*Wo, k*k*C), order (kh, kw, c)."""
    N, H, W, C = x.shape
    v = np.lib.stride_tricks.sliding_window_view(x, (k, k), axis=(1, 2))  # (N,Ho,Wo,C,k,k)
    v = v.transpose(0, 1, 2, 4, 5, 3)                                      # (N,Ho,Wo,k,k,C)
    Ho, Wo = H - k + 1, W - k + 1
    return v.reshape(N, Ho * Wo, k * k * C), Ho, Wo

def requant_relu(acc, M, S):
    v = (acc * M + (1 << (S - 1))) >> S     # round, then arithmetic shift
    return np.clip(v, 0, 127)               # ReLU (low clamp at 0) + int8 high clamp

def maxpool2(x):
    N, H, W, C = x.shape
    Ho, Wo = H // 2, W // 2
    return x[:, :2*Ho, :2*Wo, :].reshape(N, Ho, 2, Wo, 2, C).max(axis=(2, 4))

def int8_forward(x_q):
    N = x_q.shape[0]
    x = x_q.reshape(N, 28, 28, 1)

    p1, H1, W1o = im2col(x)                       # (N, 676, 9)
    acc1 = p1 @ W1_mat                             # ACCELERATOR does this
    r1 = requant_relu(acc1 + b1_q, M1, SHIFT).reshape(N, H1, W1o, 8)
    a1 = maxpool2(r1)                              # (N,13,13,8)

    p2, H2, W2o = im2col(a1)                       # (N, 121, 72)
    acc2 = p2 @ W2_mat                             # ACCELERATOR does this
    r2 = requant_relu(acc2 + b2_q, M2, SHIFT).reshape(N, H2, W2o, 8)
    a2 = maxpool2(r2)                              # (N,5,5,8)

    f = a2.reshape(N, -1)                          # (N,200)
    acc3 = f @ W3_mat                              # ACCELERATOR does this
    scores = acc3 + b3_q
    pred = np.argmax(scores, axis=1)               # first max wins on ties
    return dict(acc1=acc1, a1=a1, acc2=acc2, a2=a2, acc3=acc3,
                scores=scores, pred=pred)

In [8]:
# int8 accuracy on all 10,000 test images
pred_all = int8_forward(quant_input(test_images))["pred"]
int8_acc = float(np.mean(pred_all == test_labels))
print('INT8 test accuracy (10,000 images):', int8_acc)

# the 100 images that will run in Xcelium
x100 = quant_input(test_images[:N_TEST])
y100 = test_labels[:N_TEST]
ref  = int8_forward(x100)
acc100 = float(np.mean(ref["pred"] == y100))
float_pred100 = np.argmax(model.predict(test_images[:N_TEST], verbose=0), axis=1)
agree100 = float(np.mean(float_pred100 == ref["pred"]))
print(f'INT8 accuracy on first {N_TEST}: {acc100}   float/int8 agreement: {agree100}')

# accelerator accumulators are 32-bit: make sure nothing overflows
for k in ("acc1", "acc2", "acc3"):
    print(k, "max |value| =", np.abs(ref[k]).max())
    assert np.abs(ref[k]).max() < 2**31, k + " overflows 32 bits"

INT8 test accuracy (10,000 images): 0.9757
INT8 accuracy on first 100: 1.0   float/int8 agreement: 1.0
acc1 max |value| = 64199
acc2 max |value| = 64733
acc3 max |value| = 59346


In [9]:
os.makedirs(OUT_DIR, exist_ok=True)

def write_hex(name, arr, bits):
    """One value per line, two's complement hex. Read in SV with $readmemh."""
    mask, digits = (1 << bits) - 1, bits // 4
    with open(os.path.join(OUT_DIR, name), "w") as f:
        for v in np.asarray(arr).reshape(-1):
            f.write(f"{int(v) & mask:0{digits}x}\n")

def pad_to(a, rows=None, cols=None):
    r = a.shape[0] if rows is None else rows
    c = a.shape[1] if cols is None else cols
    out = np.zeros((r, c), dtype=np.int64)
    out[:a.shape[0], :a.shape[1]] = a
    return out

# Weights (padded so every matrix splits into 4x4 tiles)
write_hex("w1.hex", pad_to(W1_mat, rows=12), 8)     # 12 x 8   (K: 9 -> 12)
write_hex("b1.hex", b1_q, 32)                       # 8
write_hex("w2.hex", W2_mat, 8)                      # 72 x 8
write_hex("b2.hex", b2_q, 32)                       # 8
write_hex("w3.hex", pad_to(W3_mat, cols=12), 8)     # 200 x 12 (N: 10 -> 12)
write_hex("b3.hex", np.pad(b3_q, (0, 2)), 32)       # 12

# Inputs
write_hex("img.hex",    x100, 8)                    # 100 x 784 pixels (0..127)
write_hex("labels.hex", y100, 8)                    # 100 true digits

# Expected results for every layer (the answer key)
write_hex("exp_acc1.hex",   ref["acc1"], 32)        # 100 x 676 x 8   conv1 accelerator output
write_hex("exp_a1.hex",     ref["a1"],   8)         # 100 x 13 x 13 x 8
write_hex("exp_acc2.hex",   ref["acc2"], 32)        # 100 x 121 x 8   conv2 accelerator output
write_hex("exp_a2.hex",     ref["a2"],   8)         # 100 x 200
write_hex("exp_scores.hex", np.pad(ref["scores"], ((0, 0), (0, 2))), 32)  # 100 x 12
write_hex("exp_pred.hex",   ref["pred"], 8)         # 100 predicted digits

print(sorted(os.listdir(OUT_DIR)))

['b1.hex', 'b2.hex', 'b3.hex', 'exp_a1.hex', 'exp_a2.hex', 'exp_acc1.hex', 'exp_acc2.hex', 'exp_pred.hex', 'exp_scores.hex', 'img.hex', 'labels.hex', 'w1.hex', 'w2.hex', 'w3.hex']


In [10]:
# Constants for the testbench
with open(os.path.join(OUT_DIR, "mnist_params.svh"), "w") as f:
    f.write("// Generated by Colab export - do not edit\n")
    f.write(f"localparam int N_IMG = {N_TEST};\n")
    f.write(f"localparam int SHIFT = {SHIFT};\n")
    f.write(f"localparam longint M1 = {M1};\n")
    f.write(f"localparam longint M2 = {M2};\n")

# Summary of everything measured in Python
summary = dict(
    float_acc_10k=float(float_acc), int8_acc_10k=int8_acc,
    int8_acc_first100=acc100, float_int8_agreement_first100=agree100,
    M1=M1, M2=M2, SHIFT=SHIFT,
    max_abs_acc=[int(np.abs(ref[k]).max()) for k in ("acc1", "acc2", "acc3")],
)
with open(os.path.join(OUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
model.save(os.path.join(OUT_DIR, "mnist_cnn_float.keras"))
print(json.dumps(summary, indent=2))

# Zip and download
zip_name = OUT_DIR + ".zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(OUT_DIR)):
        z.write(os.path.join(OUT_DIR, fn))
from google.colab import files
files.download(zip_name)

{
  "float_acc_10k": 0.9764999747276306,
  "int8_acc_10k": 0.9757,
  "int8_acc_first100": 1.0,
  "float_int8_agreement_first100": 1.0,
  "M1": 33088,
  "M2": 32130,
  "SHIFT": 24,
  "max_abs_acc": [
    64199,
    64733,
    59346
  ]
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>